## TIFON Database. Simulation's folders (CODA ring)

**Prefacio de lo que viene**

De todos los tipos de anillos que FRODO puede llevar encima, este es el más completo y del que más orgulloso estoy (agradecimientos a TIFON por ponerme este muerto encima).

Para quien no lo sepa, CODA es un solver de CFD diseñado conjuntamente por Airbus S.A y el DLR (Deutsches Zentrum für Luft- und Raumfahrt -- German Aerospace Center). Programado en C y con una interfaz para python, la Santa Casa tiene el *honor* de ser co-desarrolladora de este software, aportando talento y visión de futuro a un programa emergido del infierno. Airbus siempre apuesta por el conocimiento propio y el know-how de la empresa, focalizando en el aprendizaje autónomo (autodidactas) y las formas de trabajo más difíciles que hacen de sus empleados (y subordinados) auténticos supervivientes intelectuales.

Aquellos que no se consideren dignos de estas habilidades o simplemente no tengan el MUIA, pueden acudir al único manual de ayuda que se proporciona, jonatan.nunez@upm.es, o en su defecto a una ayuda rápida, miguel.jaraizga@upm.es. En ambos casos, el usuario asume la completa responsabilidad de la validez de los conocimientos proporcionados por las fuentes de ayuda, así como de los resultados obtenidos de ellas, las consecuencias sobre los cálculos, informes, estudios, puestos de trabajo y consecuencias legales o penales.

**Estructura de datos**

El formato CODA espera encontrar un directorio (root_dir) con la siguiente estructura de datos:
- ./metadata/
    - ./metadata/cases_metadata.json
    - ./metadata/df_cases.csv
- ./outputs/
    - ./outputs/param1\_\<param1_value1\>\_param2\_\<param2_value1\>
    - ./outputs/param1\_\<param1_value2\>\_param2\_\<param2_value2\>
    - ...

El archivo json de la carpeta metadata ha sido generado amablemente por MERRY y PIPPIN. En él se especifican cosas tan subnormales pero necesarias como:
- folder\_fmt: Es el esquema del nombre explicativo de las carpetas (p.e. aoa\_\<aoa\_value\>\_mach\_\<mach\_value\>\_re\_\<re\_value\>).
- eq\_type: Ecuaciones que se plantean resolver. Te reirás, pero cuando pasas de RANS a LES es como pasar de disparar con un tirachinas en el colegio a estar en Iraq con un sniper.
- design\_vars: nombre de los parámetros que definen cada simulación. Obviamente, deben coincidir con los encontrados en folder\_fmt. Si no es así, ya te digo yo que no es culpa de Merry y Pippin.
- df\_cases: dataframe con los valores de los parámetros utilizados. Es básicamente el espacio de diseño.
- num_stages: Número de etapas que se han programado

El archivo df_cases.csv es una replica del dato guardado en el json pero adaptable a los humanos. Pura redundancia, nada más. Está pensado para enviar o mostrar en presentaciones.

Es importante recalcar que en la carpeta *./metadata/* iremos guardando los archivos .csv que contendrán un resumen de los resultados obtenidos. Porfa, no borres la carpeta porque creas que estorba. Más estorba el tráfico por la mañana y aun así venimos a *aportar*.

En la carpeta *./outputs/* encontramos todas las carpetas de los casos, las cuales serán analizadas por FRODO (siempre con la ayuda de SAM).

*¿Y qué pasa si no tengo el archivo *cases_metadata.json*?*

Pues nada colega, que habrá que improvisar. En tal caso, FRODO intentará adivinar el nombre y valor de los parámetros utilizados. ¿Funciona la mayoría de las veces? Sí, pero solo si no tienes más tipos de carpetas dentro de root\_dir y si no hay cosas raras en el nombre. De nada por el esfuerzo.


### Lectura y estado

In [ ]:
# Lectura normal.
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

# initial_parse es True por defecto. Hace un escaneo inicial de las carpetas y genera la primera versión de sim_metadata.
# Tenlo siempre activado porfa, a no ser que sepas lo que estás haciendo.
datafolder = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1'

db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)

# Ya sabes que FRODO es un paquete con cualquier cosa más violenta que matar una mosca, así siempre es recomendable llevar algo de seguridad:

from FotR import LEGOLAS
your_elf = LEGOLAS(db)


#### Info básica

In [ ]:
# Cuando se disponen de las carpetas de simulación, FRODO permite imprimir la información exportada de cado de los dominios definidos.
# Esto es útil para saber si todos son comunes a todas las fronteras y etapas, o si por el contrario tenemos que excluir alguna en los extract_inputs/outputs
# db.reader.print_available_cadgroup_ids(stage=0, vtu_type = 'surface')

# db.reader.print_available_cadgroup_ids(stage=1, vtu_type = 'volume')


# Todos los casos en CODA están numerados por orden del primer parámetro leído (en este caso, el Aoa).
# Ese orden es el que se sigue en df_state y sirve de referencia para los scatter plots siguientes.
# Si queremos conocer el caso por su índice, nos basta con hacer: 
for idx in [82, 5, 41]:
    print(f'Nombre del caso {idx}: ', db.reader.case_per_idx(idx))

In [ ]:
# ¿Cómo está nuestra simu favoritaaaaaa? Podemos pedirle a Légolas que nos muestre las stage en el espacio de diseño.
# Sólo podremos pintar cuando tengamos un número de parámetros menor o igual a 3:
your_elf.state.plot()
your_elf.params.plot_space(color="stage")


#### Detalles puntuales

In [ ]:
# Este diccionario es propio del formato CODA. Guarda la información que tenemos en el json para utilizarla en múltiples funciones.

SAM.DictVisualizer.rich_tree(db.metadata)

In [ ]:
# SAM es un bonachón, y si estamos enfadados con él por lo que nos dice siempre lo va a intentar de otra forma, pero seguirá siendo la misma verdad.
# En este caso, te muestro otra de las formas de enseñar la realidad de un diccionario. Te aviso de que este sim_metadata es tocho, por lo que te lo dejo preparado para imprimir.
SAM.DictVisualizer.pretty_print(db.sim_metadata, depth=5, output_file='./sim_metadata_example.txt')


In [ ]:
# Todas mis simulaciones de CODA exportan dos vtu dedicados a los humanos ineptos que no somos capaces de leer binario y nos encantan los colorines.
# Por si quieres ver qué tiene dentro un caso a lo burro, puedes usar esto:
path = db.sim_metadata[db.reader.case_per_idx(3)]['path']
mesh = db.reader.load_vtu_from_stage(case_name=db.reader.case_per_idx(3), stage=0, vtu_type='surface', verbose=True)
print(list(mesh.cell_data.keys()))

### Diccionario resumen de los datos: data_dict

FRODO, como buen milenial que es, no está cubierto ni por la seguridad social, así que, para evitar el estrés excesivo, necesita resumir todos los datos que tiene en diccionarios. El principal para el postproceso posterior de los datos es data_dict. Este diccionario se construye extrayendo los datos de las mallas en dos grandes grupos: inputs y outputs.



#### Caso de volumen

In [ ]:
db.extract_inputs(id_groups=(4,), vtu_type='volume', cases_idx = 'all')

# Podemos definir qué cosas queremos ignorar para los datasets futuros. Si no sabes qué puedes elegir, te recuerdo que hay un método que permite resumir las variables por cada CADGroup:
# Ejemplo: db.reader.print_available_cadgroup_ids(stage=1, vtu_type = 'volume')
fuera = ['GlobalNumber', 'CADGroupID', 'AugState_EnergyStagnationDensity',
         'AugState_PressureEffective', 'AugState_SpecificHeatPressure', 'AugState_Temperature',
         'AugState_ThermalConductivity', 'AugState_ThermalConductivityEffective', 'AugState_TurbulentSANuTildeDensity',
         'AugState_ViscosityEffective', 'AugState_ViscosityMolecular', 'AugStateGrad_DensityGradient',
         'AugStateGrad_EnergyStagnationDensityGradient', 'AugStateGrad_MomentumXGradient', 'AugStateGrad_MomentumYGradient',
         'AugStateGrad_MomentumZGradient', 'AugStateGrad_PressureEffectiveGradient', 'AugStateGrad_SpecificHeatPressureGradient',
         'AugStateGrad_TemperatureGradient', 'AugStateGrad_ThermalConductivityGradient', 'AugStateGrad_TurbulentSANuTildeDensityGradient', 
         'AugStateGrad_ViscosityMolecularGradient', 'State_Density', 'State_EnergyStagnationDensity', 'State_Momentum', 'State_TurbulentSANuTildeDensity']

db.extract_outputs(id_groups=(4,), stage=1, vtu_type='volume', var_name_excluded = None)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
radio=0.3
iter=10
case=1
stage=0
coord = file['Coord'][file['idx_sort'][stage,case, :]]
mask_coord = np.linalg.norm(coord[:, [0,2]], axis=1) < radio
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sc0 = ax.scatter(
    coord[mask_coord, 0][::iter], coord[mask_coord, 2][::iter],
    c=np.linalg.norm(file['Vars'][str(stage)]['State_Momentum'][:, file['idx_sort'][stage, case, :], case], axis=0)[mask_coord][::iter],
    cmap='viridis', s=1,
    norm='symlog'
    )
cbar = plt.colorbar(sc0, ax=ax, label='Momentum')
cbar.ax.set_yscale('log')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')


#### Caso de superficie

In [ ]:
case_idx = list(range(100))
fuera = [64, 79, 87, 88, 94]
for c in fuera:
    case_idx.remove(c)

db.extract_inputs(
    id_groups = (3,),
    cases_idx = case_idx,
    vtu_type='surface',
    verbose=False
    )

db.extract_outputs(
    id_groups=(3,),
    stage=0, cases_idx = case_idx,
    var_name_excluded = [
        'BoundaryValues_CoefSkinFrictionX',
        'BoundaryValues_CoefSkinFrictionY',
        'BoundaryValues_CoefSkinFrictionZ'
        ],
    vtu_type='surface',
    )

db.extract_outputs(
    id_groups=(3,),
    stage=1, cases_idx = case_idx,
    var_name_excluded = [
        'BoundaryValues_CoefSkinFrictionX',
        'BoundaryValues_CoefSkinFrictionY',
        'BoundaryValues_CoefSkinFrictionZ'
        ],
    vtu_type='surface',
    )
# file = db.data_dict['CADGroup_3']

In [ ]:
file = db.data_dict['CADGroup_3']
case=0
stage='1'
c2 = file['Vars'][stage]['BoundaryValues_CoefPressure'][:,case]
factor=7
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sc0 = ax.scatter(
    file['Coord'][:, 0], file['Coord'][:, 2],
    c='black',
    s=1,
)
ax1 = ax.twinx()

sc1 = ax1.scatter(
    file['Coord'][:, 0], c2,
    c='red',
    s=1)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.set_title(f'Case {case}, {db.metadata["design_vars"][0]}: {db.data_dict["CADGroup_3"]["FlCc"][case,0]:.2f}, {db.metadata["design_vars"][1]}: {db.data_dict["CADGroup_3"]["FlCc"][case,1]:.2f}')
ax.set_ylim(bottom=np.min(file['Coord'][:, 2])*factor, top=np.max(file['Coord'][:, 2])*factor)
ax1.set_ylabel('BoundaryValues_CoefPressure')


In [ ]:
db.summary_data()

#### Caso mixto

In [ ]:
db.extract_inputs(id_groups=(4,), vtu_type='volume', cases_idx = [3, 18])
db.extract_inputs(id_groups=(3,), vtu_type='surface', cases_idx = [3, 18])
db.extract_outputs(id_groups=(4,), stage=1, vtu_type='volume', var_name_excluded = None)


### Residuos

**El apartado de autocrítica**

Decía un entrenador mío que lo que no se mide no se puede mejorar. Y en cierta forma, es una enseñanza universal que me llevé para la vida. Por lo tanto, para poder juzgar cualquier cosa, necesitas métricas y matemáticas. El hecho de que te conformes con ello o no, ya es decisión tuya.

Como ya hemos dicho anteriormente, CODA es el único formato de anillo que actualmente permite autocrítica (mostrado de residuos). Vamos a analizar brevemente las funciones más importantes con la base de ejemplo anterior.

Todos los formatos de anillo tendrán a su disposición una subclase dedicada en FRODO.RESIDUALS, la cual recoge las variables internas que calculemos y las funciones específicas.

Analizaremos solo FRODO.RESIDUALS.CODAResiduals, como hemos mencionado

In [ ]:
your_elf.residuals.plot_case(case_idx=82, mode="scaled")

In [ ]:
# Unos resúmenes guapos de las variables integrales siempre está guay para mostrar.
# Recuerda: Las variables integrales son aquellas impresas en los archivos *_wall_boundary_integrals.dat
df_post = db.residuals.get_df_metrics(var_metrics=['CoefLift', 'CoefDrag', 'CoefMomentY'], iter_var=1000, save=True)
display(df_post)

# Este csv guarda los valores finales de los residuos y variables integrales de cada caso por etapa. Para que lo pongas en las ppt de Airbus.

In [ ]:
db.residuals.plot_state_calculation(num_stages=2)

In [ ]:
# Otra forma de visualizar los valores integrales es mediante el siguiente método:
results_mean, results_std = db.residuals.integrals_convergence_criteria(
    iterations_back=1000,
    only_finished=False,
    only_converged=False,
    mode='3D',
    plot=True, # a pesar de que nos da siempre la media y la std, podemos verlo gráficamente (siempre y cuando sean tres o menos parámetros)
    verbose=True
)

# Los resultados de esta

In [ ]:
db.residuals.plot_all_final_residuals(mode='scaled', stage='all', only_finished=False, activate_idx = True)

### Creación de DataSets y exportar datos

Una vez hemos decidido cuáles van a ser nuestras variables y parámetros con los que trabajar, podemos empezar a generar los Datasets. También podemos hacer cálculos adicionales o análisis previos antes de exportar, para enriquecer más que nada y poder presentar algo decente

In [ ]:
# Como primera operación que podemos realizar, se pueden añadir manualmente datos extra a data_dcit, en el apartado de Aux.
# Esto nos permite enriquecer el dataset con variables externas o calculadas por nosotros mismos.

# Es obligatorio que estén los datos asociados a un CADGroup. Recuerda que puedes ver el resultado imprimiendo de nuevo el data_dict con SAM.

db.sets.add_to_data_dict(arr=db.data_dict['CADGroup_3']['Coord'], id_group='4', key_location='Airfoil')

In [ ]:
# Campos
your_elf.fields.select_group("CADGroup_4", stage=stage)
your_elf.fields.plot_field("State_Density", case_idx=0, group_key="CADGroup_4", stage=stage, coord_idx=(0, 2))


#### Dataset a formato NUMPYFILE

Sip, oficialmente este es el primer crossover de la librería. Se pueden exportar los datos recopilados de un CADGroup a formato npy, tal y como se concibió antaño la base de datos Aero-Nef ([enlace web](https://www.nature.com/articles/s41598-024-76983-w.pdf))

In [ ]:
SAM.DictVisualizer.rich_tree(db.data_dict)
db.sets.save_to_npy(case_idx='all', stage=1, id_group='4', filepath='./cases_3_18_stage_1.npy', verbose=True)

#### Dataset a formato h5

In [ ]:
db.sets.save_to_h5(filepath='./dataset_prueba.h5', verbose=True)

#### Dataset a formato pyLOM

Para los amantes del compañerismo, y los puristas del proyecto TIFON, tenemos aquí a la ensecia de almacenamiento de datos. Podemos exportar los datos recopilados en data_dict al formato pyLOM, ya ordenados y listos para su uso. Si no mencionamos nada en variables externas (external_vars), FRODO cargará solo los parámetros de diseño en el diccionario interno de parámetros.

In [ ]:
[d] = db.sets.create_NN_pylom(id_groups='3', stage=stage, idx_to_print='all', external_vars=None, save_path='./')

Aunque también podemos asignar más manualmente. Por ejemplo, se han escogido aquí unos cuantos de los que se calculan en el dataframe que resumen la clase residuals, df_post. En concreto, esta ha sido la manera de exportar la base de datos para el proyecto TIFON.

In [ ]:
df_post = db.residuals.get_df_metrics(
    var_metrics=['CoefLift', 'CoefDrag', 'CoefMomentY'], # Variables que deben de estar en los archivos *__monitors_wall_boundary_integrals.dat
    iter_var=1000,  # Últimas iteraciones sobre las que calculamos la varianza de las variables integrales.
    save=True   # Puede guardar automáticamente en el directorio metadata de root_dir.
    )
df_post.sort_values(by=['aoa', 'mach'], inplace=True)
# case_idx = list(range(100))
# fuera = list(range(50)) + [64, 79, 87, 88, 94]
# for c in fuera:
#     case_idx.remove(c)
df_post_filtered = df_post[df_post['case_idx'].isin(case_idx)]
# print(df_post_filtered)

In [ ]:
d = []
for stage, factor in zip([0, 1], [1, 10]):
    vars = {
        'aoa' : {'idim':0, 'value':df_post_filtered['aoa'].values}, # Angle of attack
        'M'   : {'idim':0, 'value':df_post_filtered['mach'].values},   # Mach number
        'Re'  : {'idim':0, 'value':df_post_filtered['Re'].values},  # Reynolds
        'CL'  : {'idim':0, 'value':df_post_filtered[f'CoefLift_mean_stage{stage}'].values / factor},  # Mean Lift coefficient
        'CD'  : {'idim':0, 'value':df_post_filtered[f'CoefDrag_mean_stage{stage}'].values / factor},  # Mean Drag coefficient
        'CMy'  : {'idim':0, 'value':df_post_filtered[f'CoefMomentY_mean_stage{stage}'].values / factor},  # Mean Momentum coefficient
        'varCL'  : {'idim':0, 'value':df_post_filtered[f'CoefLift_var_stage{stage}'].values},  # Var Lift coefficient
        'varCD'  : {'idim':0, 'value':df_post_filtered[f'CoefDrag_var_stage{stage}'].values},  # Var Drag coefficient
        'varCM'  : {'idim':0, 'value':df_post_filtered[f'CoefMomentY_var_stage{stage}'].values},  # Var Momentum coefficient
    }
    d.append(db.sets.create_NN_pylom(id_groups='3', stage=stage, idx_to_print='all', external_vars=vars, save_path='./'))

In [ ]:
import sys, numpy as np
import matplotlib.pyplot as plt
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
import pyLOM
FONTSIZE = 12
file = "/home/airbus/CETACEO_cp_interp/CADGroup_3_stage_1.h5"
d  = pyLOM.Dataset.load(file)
 
cp = d["BoundaryValues_CoefPressure"]
x = d.xyz[:, 0]
aoa = d.get_variable('aoa')
mach = d.get_variable('M')
N_airfoil = len(d.xyz)
 
aoa_82 = 4.08
mach_82 = 1.45
 
tol = 1e-2
idx = np.where((np.abs(aoa - aoa_82) < tol) & (np.abs(mach - mach_82) < tol))[0][0]
print(f"Index of case with AoA={aoa_82} deg and Mach={mach_82}: {idx}")
 
fig, ax = plt.subplots(figsize=(7,7))
ax.scatter(x, cp[:,idx], color='blue', label=f'Case {idx}', s=5)
ax.set_xlabel('x/c [-]', fontsize=FONTSIZE)
ax.set_ylabel('Cp [-]', fontsize=FONTSIZE)

## TIFON Database. Isolated cases (NUMPYFILE ring)

In [ ]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

datafolder = '/home/airbus/CETACEO_cp_interp/DATA/TIFON'

db = FRODO(root_dir=datafolder, format = 'NUMPYFILE', file=['cases_3_18_stage_1.npy'])

In [ ]:
SAM.DictVisualizer.rich_tree(db.npy_dict)

In [ ]:
example = db.npy_dict['cases_3_18_stage_1.npy']['State_Density']
a = example[0, 100]
print(type(a))